# 07 — Hedefli Veri Toplama
gap_report.csv'deki bosluklari doldur. Her tur >= 2000 filme ulasana kadar targeted spider calistir.

In [ ]:
import ast
import os
import shutil
import subprocess
from collections import Counter
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# --- Yerel ---
CODE_ROOT   = Path('../')
DATA_ROOT   = Path('../')

# --- Colab ---
# from google.colab import drive
# drive.mount('/content/drive')
# CODE_ROOT = Path('/content/drive/MyDrive/film-genre-project')
# DATA_ROOT = Path('/content/drive/MyDrive/film-genre-project-data')

SCRAPY_DIR    = CODE_ROOT / 'src' / 'scraper'
IDS_FILE      = CODE_ROOT / 'tmdb_movie_ids.txt'
LABELS_CSV    = DATA_ROOT / 'labels.csv'
LABELS_V2     = DATA_ROOT / 'labels_v2.csv'
GAP_REPORT    = DATA_ROOT / 'gap_report.csv'
POSTERS_DIR   = DATA_ROOT / 'posters'

print('SCRAPY_DIR   :', SCRAPY_DIR.exists())
print('IDS_FILE     :', IDS_FILE.exists())
print('LABELS_CSV   :', LABELS_CSV.exists())
print('GAP_REPORT   :', GAP_REPORT.exists())

## 1. Mevcut Bosluklar

In [ ]:
gap_df = pd.read_csv(GAP_REPORT)
singles = gap_df[gap_df['type'] == 'single'].sort_values('deficit', ascending=False)

print('Tur           Mevcut  Hedef  Acik')
print('-' * 45)
total_deficit = 0
for _, row in singles.iterrows():
    mark = '<-- EKSIK' if row['deficit'] > 0 else 'tamam'
    print(f"  {row['combination']:20s} {int(row['current_count']):5,}  {int(row['target_count']):5,}  {int(row['deficit']):5,}  {mark}")
    total_deficit += row['deficit']

print('-' * 45)
print(f'Minimum ihtiyac : {int(total_deficit):,} film')
print(f'Onerilen hedef  : {int(total_deficit * 1.3):,} film (+%%30 ortusme payi)')

## 2. Test Cekim (300 film)
Spider'in dogru calistığını dogrula — kabul ettigi filmler hedef turlere ait mi?

In [ ]:
# Kac satir var baslamadan once
with open(LABELS_CSV, encoding='utf-8') as f:
    before_count = sum(1 for _ in f) - 1  # header haric
print(f'Cekim oncesi labels.csv: {before_count:,} film')

cmd = [
    'scrapy', 'crawl', 'targeted',
    '-s', f'IDS_FILE={IDS_FILE.resolve()}',
    '-s', f'LABELS_PATH={LABELS_CSV.resolve()}',
    '-s', f'GAP_REPORT_PATH={GAP_REPORT.resolve()}',
    '-s', f'IMAGES_STORE={POSTERS_DIR.resolve()}',
    '-s', 'CLOSESPIDER_ITEMCOUNT=300',
    '--logfile', str(CODE_ROOT / 'targeted_test.log'),
]

print('Komut:', ' '.join(str(c) for c in cmd))
result = subprocess.run(cmd, cwd=str(SCRAPY_DIR), capture_output=True, text=True)
print(result.stdout[-3000:] if result.stdout else '')
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

In [ ]:
# Kac yeni film eklendi?
with open(LABELS_CSV, encoding='utf-8') as f:
    after_count = sum(1 for _ in f) - 1
new_films = after_count - before_count
print(f'Eklenen film sayisi : {new_films}')
print(f'Toplam labels.csv   : {after_count:,}')

# Son eklenen filmlerin tur dagilimi
df_all = pd.read_csv(LABELS_CSV, dtype={'tmdb_id': str})
df_all['genres'] = df_all['genres'].apply(
    lambda x: x.split('|') if isinstance(x, str) else ast.literal_eval(str(x))
)
new_films_df = df_all.tail(new_films)
new_genre_counts = Counter(g for gs in new_films_df['genres'] for g in gs)

print('\nYeni filmlerin tur dagilimi:')
for g, c in sorted(new_genre_counts.items(), key=lambda x: -x[1]):
    print(f'  {g:20s}: {c}')

## 3. Tam Cekim
Spider tum hedefler karsilaninca otomatik durur. CLOSESPIDER_ITEMCOUNT yok — guvenlik icin sadece
log dosyasi kontrol edilir.

In [ ]:
with open(LABELS_CSV, encoding='utf-8') as f:
    start_count = sum(1 for _ in f) - 1
print(f'Tam cekim basliyor. Mevcut film sayisi: {start_count:,}')

cmd_full = [
    'scrapy', 'crawl', 'targeted',
    '-s', f'IDS_FILE={IDS_FILE.resolve()}',
    '-s', f'LABELS_PATH={LABELS_CSV.resolve()}',
    '-s', f'GAP_REPORT_PATH={GAP_REPORT.resolve()}',
    '-s', f'IMAGES_STORE={POSTERS_DIR.resolve()}',
    '--logfile', str(CODE_ROOT / 'targeted_full.log'),
]

print('Komut:', ' '.join(str(c) for c in cmd_full))
result = subprocess.run(cmd_full, cwd=str(SCRAPY_DIR), capture_output=True, text=True)
print(result.stdout[-3000:] if result.stdout else '')
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

## 4. Cekim Sonrasi Tur Dagilimi

In [ ]:
TARGET_GENRES = [
    'Action', 'Adventure', 'Animation', 'Comedy', 'Crime',
    'Documentary', 'Drama', 'Family', 'Fantasy', 'History',
    'Horror', 'Mystery', 'Romance', 'Science Fiction', 'Thriller'
]

df_final = pd.read_csv(LABELS_CSV, dtype={'tmdb_id': str})
df_final['genres'] = df_final['genres'].apply(
    lambda x: x.split('|') if isinstance(x, str) else ast.literal_eval(str(x))
)
df_final['genres'] = df_final['genres'].apply(
    lambda gs: [g for g in gs if g in TARGET_GENRES]
)
df_final = df_final[df_final['genres'].map(len) > 0]

genre_counts = Counter(g for gs in df_final['genres'] for g in gs)

print(f'Toplam film: {len(df_final):,}')
print()
print('Tur           Sayi   Durum')
print('-' * 40)
for g in TARGET_GENRES:
    c = genre_counts.get(g, 0)
    target = 2500 if g in ('Drama', 'Comedy') else 2000
    status = 'OK' if c >= target else f'EKSIK ({target - c})'
    print(f'  {g:20s}: {c:5,}  {status}')

# Bar chart
genres_sorted = sorted(TARGET_GENRES, key=lambda g: genre_counts.get(g, 0))
counts_sorted = [genre_counts.get(g, 0) for g in genres_sorted]
colors = ['#2ecc71' if c >= 2000 else '#e74c3c' for c in counts_sorted]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(genres_sorted, counts_sorted, color=colors, alpha=0.85)
ax.axvline(2000, color='navy', linestyle='--', linewidth=1.5, label='Hedef: 2,000')
ax.axvline(2500, color='orange', linestyle=':', linewidth=1.5, label='Downsample: 2,500')
ax.set_xlabel('Film Sayisi')
ax.set_title('Cekim Sonrasi Tur Dagilimi')
ax.legend()
plt.tight_layout()
plt.show()

## 5. labels_v2.csv Olustur
labels.csv'nin anlık snapshotunu labels_v2.csv olarak kaydet. Phase 8 buradan calisacak.

In [ ]:
shutil.copy(LABELS_CSV, LABELS_V2)

with open(LABELS_V2, encoding='utf-8') as f:
    v2_count = sum(1 for _ in f) - 1

print(f'labels_v2.csv olusturuldu: {LABELS_V2}')
print(f'Toplam film: {v2_count:,}')